# Thermal Generation Prediction — Feature Selection & Model Comparison

本 Notebook 是前一版德国电力项目的预测模块。数据口径保持一致：

- 元数据：`curve_id`、`Technology_is`、`Technology_Fuel_is`
- 小时宽表：`reference_time_start` + 各 `curve_id` 发电序列
- Thermal 为预测目标；Wind、Solar 及时间信息为预测输入
- 允许使用时点 t 已知的 Wind_t、Solar_t；不使用 Thermal 的任何特征

## 实验设计

**Experiment 1 — Feature-set comparison**
1. 构造约 200 个候选特征。
2. 只用 Train 计算特征预测力（与 y 的绝对 Pearson correlation）。
3. 按预测力排序，使用 Train 特征相关矩阵做贪心去相关，保留最多 100 个。
4. 构建嵌套的 Small / Medium / Large 因子集（最多 20 / 50 / 100 个）。
5. 固定 Ridge，比较三个因子集的 Validation 表现。

**Experiment 2 — Model comparison**
1. 根据 Experiment 1 的 Validation RMSE 选择一个因子集。
2. 固定这个因子集，比较 5 个 CPU 模型。
3. 每个模型使用小规模参数候选，在 Validation 上选择参数。
4. 只对最终选出的模型报告一次 Test 表现。

所有划分均按时间顺序进行，不随机打乱。相关性筛选、缺失值填补和调参均只使用 Train / Validation，不使用 Test。

## 0. Imports and data paths

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    HistGradientBoostingRegressor, GradientBoostingRegressor
)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid

RANDOM_STATE = 42
DATA_DIR = Path(".")
SERIES_PATH = DATA_DIR / "generation_series.csv"
META_PATH = DATA_DIR / "curve_metadata.csv"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)

## 1. Load and aggregate the standard dataset

In [ ]:
series = pd.read_csv(SERIES_PATH)
meta = pd.read_csv(META_PATH)

series.columns = series.columns.astype(str).str.strip()
meta.columns = meta.columns.astype(str).str.strip()

time_col = "reference_time_start"
for col in ["curve_id", "Technology_is", "Technology_Fuel_is"]:
    if col not in meta.columns:
        raise ValueError(f"Metadata missing required column: {col}")
if time_col not in series.columns:
    raise ValueError(f"Series file must contain {time_col}")

series[time_col] = pd.to_datetime(series[time_col], utc=True, errors="coerce")
series = (series.dropna(subset=[time_col])
                .sort_values(time_col)
                .drop_duplicates(time_col, keep="last")
                .set_index(time_col))

meta["curve_id"] = meta["curve_id"].astype(str).str.strip()
matched = [c for c in series.columns if c in set(meta["curve_id"])]
if not matched:
    raise ValueError("No curve_id in metadata matches columns in generation_series.csv.")

series[matched] = series[matched].apply(pd.to_numeric, errors="coerce")
meta["Technology_is"] = meta["Technology_is"].astype("string").str.strip()
meta["Technology_Fuel_is"] = meta["Technology_Fuel_is"].astype("string").str.strip()

def tech_group(x):
    x = "" if pd.isna(x) else str(x).lower()
    # Aggregate 'Power generation' curves are excluded to avoid double counting.
    if "power generation" in x:
        return "Aggregate"
    if "thermal power" in x:
        return "Thermal"
    if "wind turbines" in x or "wind" in x:
        return "Wind"
    if "solar" in x:
        return "Solar"
    if "pump storage" in x:
        return "PumpedStorage"
    if "nuclear" in x:
        return "Nuclear"
    if "reservoir" in x or "run of river" in x or "hydro" in x:
        return "Hydro"
    return "Other"

meta["source"] = meta["Technology_is"].map(tech_group)

# Reindex to a continuous hourly UTC index; gaps remain NaN rather than being filled with zero.
full_index = pd.date_range(series.index.min(), series.index.max(), freq="h", tz="UTC")
series = series.reindex(full_index)
series.index.name = time_col

hourly = {}
for group in ["Thermal", "Wind", "Solar", "PumpedStorage", "Hydro", "Nuclear", "Other"]:
    ids = meta.loc[meta["source"].eq(group), "curve_id"].tolist()
    ids = [c for c in ids if c in series.columns]
    if ids:
        hourly[group] = series[ids].sum(axis=1, min_count=1)

hourly = pd.DataFrame(hourly)
print("Aggregated data shape:", hourly.shape)
print("Date range:", hourly.index.min(), "to", hourly.index.max())
display(hourly.head())
display(meta[["curve_id", "Technology_is", "Technology_Fuel_is", "source"]])

## 2. Define target, predictors and chronological Train / Validation / Test

In [ ]:
required = ["Thermal", "Wind", "Solar"]
missing = [c for c in required if c not in hourly.columns]
if missing:
    raise ValueError(f"Required source series not found: {missing}. Check metadata mapping.")

# German local calendar features; the generation values remain indexed in UTC.
local_index = hourly.index.tz_convert("Europe/Berlin")
base = pd.DataFrame(index=hourly.index)
base["Thermal"] = hourly["Thermal"]
base["Wind"] = hourly["Wind"]
base["Solar"] = hourly["Solar"]

# Calendar fields are known at time t.
base["Hour"] = local_index.hour
base["Weekday"] = local_index.dayofweek
base["Month"] = local_index.month
base["DayOfYear"] = local_index.dayofyear
base["IsWeekend"] = (local_index.dayofweek >= 5).astype(int)
base["IsDaylightHour"] = ((local_index.hour >= 6) & (local_index.hour < 20)).astype(int)
base["IsSummer"] = local_index.month.isin([5, 6, 7, 8, 9]).astype(int)

# Keep the complete hourly index so lags and rolling windows represent clock hours.
n = len(base)
i_train, i_val = int(n * 0.70), int(n * 0.85)
train_index = base.index[:i_train]
val_index = base.index[i_train:i_val]
test_index = base.index[i_val:]

print("Train:", len(train_index), train_index.min(), "to", train_index.max())
print("Validation:", len(val_index), val_index.min(), "to", val_index.max())
print("Test:", len(test_index), test_index.min(), "to", test_index.max())
assert train_index.max() < val_index.min() < test_index.min()

## 3. Build approximately 200 manual features

Feature logic is deliberately grouped by economic meaning rather than blindly generating every statistic for every rolling window:

- Current supply: Wind / Solar level, renewable total, shares and mix.
- Dynamics: selected lags, changes, ramps and momentum.
- Local state: a small number of rolling mean / volatility / range features.
- Nonlinearity: squares, logs, square roots, ratios and Wind × Solar interactions.
- Calendar interaction: Month × Hour, Solar × Hour, Solar × Season, Wind × Hour.

Rolling statistics include the current hour because the task states that current Wind and Solar are known at time t. Lag features use only earlier observations. No Thermal-derived features are created.

In [ ]:
def make_features(data):
    X = {}
    w, s = data["Wind"], data["Solar"]
    r = w + s
    eps = 1e-6

    # 1) Current renewable supply and composition.
    X["Wind_t"] = w
    X["Solar_t"] = s
    X["Renewable_t"] = r
    X["Wind_share"] = w / (r + eps)
    X["Solar_share"] = s / (r + eps)
    X["Wind_minus_Solar"] = w - s
    X["Renewable_imbalance"] = (w - s).abs() / (r + eps)

    # 2) Selected past lags.
    for lag in [1, 3, 6, 12, 24, 48, 168]:
        X[f"Wind_lag{lag}"] = w.shift(lag)
        X[f"Solar_lag{lag}"] = s.shift(lag)
        X[f"Renewable_lag{lag}"] = r.shift(lag)

    # 3) Ramps, changes and momentum.
    for lag in [1, 3, 6, 24, 168]:
        X[f"Wind_change{lag}"] = w - w.shift(lag)
        X[f"Solar_change{lag}"] = s - s.shift(lag)
        X[f"Renewable_change{lag}"] = r - r.shift(lag)

    X["Wind_acceleration"] = w.diff() - w.diff().shift(1)
    X["Solar_acceleration"] = s.diff() - s.diff().shift(1)
    X["Renewable_acceleration"] = r.diff() - r.diff().shift(1)

    # 4) Compact rolling summaries; windows are selected, not exhaustively swept.
    for window in [6, 24, 168, 720]:
        for name, z in [("Wind", w), ("Solar", s), ("Renewable", r)]:
            roll = z.rolling(window, min_periods=max(3, window // 3))
            X[f"{name}_mean{window}"] = roll.mean()
            X[f"{name}_std{window}"] = roll.std()

    # Distribution shape and range at daily / weekly scales.
    for window in [24, 168]:
        for name, z in [("Wind", w), ("Solar", s), ("Renewable", r)]:
            roll = z.rolling(window, min_periods=max(6, window // 3))
            X[f"{name}_q10_{window}"] = roll.quantile(0.10)
            X[f"{name}_q90_{window}"] = roll.quantile(0.90)
            X[f"{name}_range_{window}"] = roll.max() - roll.min()

    # 5) Exponential levels and local volatility.
    for span in [12, 48, 168]:
        X[f"Wind_ema{span}"] = w.ewm(span=span, adjust=False).mean()
        X[f"Solar_ema{span}"] = s.ewm(span=span, adjust=False).mean()
        X[f"Renewable_ema{span}"] = r.ewm(span=span, adjust=False).mean()

    X["Wind_ramp_vol24"] = w.diff().rolling(24, min_periods=12).std()
    X["Solar_ramp_vol24"] = s.diff().rolling(24, min_periods=12).std()
    X["Renewable_ramp_vol24"] = r.diff().rolling(24, min_periods=12).std()

    # 6) Nonlinear supply transformations.
    for name, z in [("Wind", w), ("Solar", s), ("Renewable", r)]:
        X[f"{name}_square"] = z ** 2
        X[f"{name}_sqrt"] = np.sqrt(z.clip(lower=0))
        X[f"{name}_log1p"] = np.log1p(z.clip(lower=0))

    X["Wind_x_Solar"] = w * s
    X["Wind_x_Solar_share"] = w * X["Solar_share"]
    X["Solar_x_Wind_share"] = s * X["Wind_share"]
    X["Wind_over_Solar"] = w / (s + 1)
    X["Solar_over_Wind"] = s / (w + 1)
    X["Wind_sq_x_Solar"] = (w ** 2) * s / (r + 1)
    X["Solar_sq_x_Wind"] = (s ** 2) * w / (r + 1)

    # 7) Standardized supply shocks, based on trailing observations only.
    for window in [24, 168, 720]:
        for name, z in [("Wind", w), ("Solar", s), ("Renewable", r)]:
            mu = z.rolling(window, min_periods=max(6, window // 3)).mean()
            sd = z.rolling(window, min_periods=max(6, window // 3)).std()
            X[f"{name}_shock{window}"] = (z - mu) / (sd + eps)

    # 8) Cyclical calendar encoding.
    idx = data.index.tz_convert("Europe/Berlin")
    hour, weekday, month, doy = idx.hour, idx.dayofweek, idx.month, idx.dayofyear
    X["Hour_sin"] = np.sin(2*np.pi*hour/24)
    X["Hour_cos"] = np.cos(2*np.pi*hour/24)
    X["Weekday_sin"] = np.sin(2*np.pi*weekday/7)
    X["Weekday_cos"] = np.cos(2*np.pi*weekday/7)
    X["Month_sin"] = np.sin(2*np.pi*month/12)
    X["Month_cos"] = np.cos(2*np.pi*month/12)
    X["DayOfYear_sin"] = np.sin(2*np.pi*doy/365.25)
    X["DayOfYear_cos"] = np.cos(2*np.pi*doy/365.25)
    X["Hour"] = hour
    X["Weekday"] = weekday
    X["Month"] = month
    X["IsWeekend"] = (weekday >= 5).astype(int)
    X["IsDaylight"] = ((hour >= 6) & (hour < 20)).astype(int)
    X["IsSummer"] = np.isin(month, [5, 6, 7, 8, 9]).astype(int)

    # 9) Explicit calendar interactions.
    for h in [6, 9, 12, 15, 18, 21]:
        X[f"Hour_is_{h}"] = (hour == h).astype(int)
        X[f"Solar_x_hour{h}"] = s * (hour == h)
        X[f"Wind_x_hour{h}"] = w * (hour == h)

    for m in [1, 4, 7, 10]:
        X[f"Month_is_{m}"] = (month == m).astype(int)
        X[f"Solar_x_month{m}"] = s * (month == m)
        X[f"Wind_x_month{m}"] = w * (month == m)

    X["Solar_x_daylight"] = s * X["IsDaylight"]
    X["Solar_x_summer"] = s * X["IsSummer"]
    X["Solar_x_hour_sin"] = s * X["Hour_sin"]
    X["Solar_x_hour_cos"] = s * X["Hour_cos"]
    X["Wind_x_hour_sin"] = w * X["Hour_sin"]
    X["Wind_x_hour_cos"] = w * X["Hour_cos"]
    X["Solar_x_month_sin"] = s * X["Month_sin"]
    X["Solar_x_month_cos"] = s * X["Month_cos"]

    for day in range(7):
        X[f"Weekday_is_{day}"] = (weekday == day).astype(int)
    seasons = {
        "Winter": [12, 1, 2], "Spring": [3, 4, 5],
        "Summer": [6, 7, 8], "Autumn": [9, 10, 11]
    }
    for season, months in seasons.items():
        flag = np.isin(month, months).astype(int)
        X[f"Wind_x_{season}"] = w * flag
        X[f"Solar_x_{season}"] = s * flag

    return pd.DataFrame(X, index=data.index).replace([np.inf, -np.inf], np.nan)

X_raw = make_features(base[["Wind", "Solar"]])
y = base["Thermal"].rename("Thermal")

print("Candidate feature count:", X_raw.shape[1])
display(X_raw.head())

## 4. Train-only predictive-power screening and de-correlation

筛选步骤简洁且计算量较小：

1. 先删除 Train 缺失率超过 10% 的候选特征。
2. 用 Train 中位数填补缺失值。
3. 计算每个特征与 Thermal 的绝对 Pearson correlation，作为单变量预测力排序指标。
4. 按排序从高到低贪心保留特征；若候选特征与已选任一特征的绝对相关系数超过 0.95，则剔除。
5. 最多保留 100 个特征。

**注意：** 相关性筛选不是因果识别；Pearson 也可能低估非线性特征价值。这里把它作为快速去冗余方法，后续通过 Ridge 因子集对照和非线性模型比较验证。

In [ ]:
# Split feature rows by the same chronological boundaries.
X_train0 = X_raw.loc[train_index].copy()
X_val0 = X_raw.loc[val_index].copy()
X_test0 = X_raw.loc[test_index].copy()
y_train = y.loc[train_index]
y_val = y.loc[val_index]
y_test = y.loc[test_index]

# Drop rows with missing target or unavailable current renewable inputs.
def valid_rows(index, target):
    return target.notna() & base.loc[index, ["Wind", "Solar"]].notna().all(axis=1)

ok_train = valid_rows(train_index, y_train)
ok_val = valid_rows(val_index, y_val)
ok_test = valid_rows(test_index, y_test)
X_train0, y_train = X_train0.loc[ok_train], y_train.loc[ok_train]
X_val0, y_val = X_val0.loc[ok_val], y_val.loc[ok_val]
X_test0, y_test = X_test0.loc[ok_test], y_test.loc[ok_test]

# Keep features with acceptable Train coverage.
missing_rate = X_train0.isna().mean()
candidate_cols = missing_rate[missing_rate <= 0.10].index.tolist()
X_train0 = X_train0[candidate_cols]
X_val0 = X_val0[candidate_cols]
X_test0 = X_test0[candidate_cols]

# Fit imputation values on Train only, then apply to all splits.
train_medians = X_train0.median()
X_train_imp = X_train0.fillna(train_medians)
X_val_imp = X_val0.fillna(train_medians)
X_test_imp = X_test0.fillna(train_medians)

# Drop constant columns based on Train.
nunique = X_train_imp.nunique()
valid_cols = nunique[nunique > 1].index.tolist()
X_train_imp = X_train_imp[valid_cols]
X_val_imp = X_val_imp[valid_cols]
X_test_imp = X_test_imp[valid_cols]

# Predictive-power ranking: absolute Pearson correlation with target, Train only.
ic = X_train_imp.corrwith(y_train).abs().dropna().sort_values(ascending=False)
ranked_cols = ic.index.tolist()

# Greedy correlation filter. Correlation matrix is computed once on Train.
corr_matrix = X_train_imp[ranked_cols].corr().abs()
selected = []
for col in ranked_cols:
    if len(selected) >= 100:
        break
    if all(corr_matrix.loc[col, kept] <= 0.95 for kept in selected):
        selected.append(col)

selected_ic = ic.loc[selected]
print("Initial candidate features:", X_raw.shape[1])
print("After missing/constant filter:", len(valid_cols))
print("After predictive-power + de-correlation:", len(selected))
display(selected_ic.rename("abs_train_corr_with_Thermal").to_frame())

## 5. Build nested Small / Medium / Large feature sets

In [ ]:
# Nested sets: Small is contained in Medium, which is contained in Large.
feature_sets = {
    "Small": selected[:20],
    "Medium": selected[:50],
    "Large": selected[:100]
}
feature_sets = {k: v for k, v in feature_sets.items() if len(v) > 0}

print({name: len(cols) for name, cols in feature_sets.items()})
assert all(
    set(feature_sets[a]).issubset(set(feature_sets[b]))
    for a, b in [("Small", "Medium"), ("Medium", "Large")]
    if a in feature_sets and b in feature_sets
)

## 6. Experiment 1 — Compare feature sets under the same Ridge baseline

所有因子集使用相同的 Ridge Pipeline（中位数填补、标准化、Ridge），避免模型差异干扰因子集比较。Alpha 固定为 10；第一轮先比较因子信息量，不在这里同时调模型参数。

In [ ]:
def metrics(y_true, y_pred):
    return {
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

ridge_results = []
ridge_models = {}

for name, cols in feature_sets.items():
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        Ridge(alpha=10.0)
    )
    model.fit(X_train_imp[cols], y_train)
    pred_val = model.predict(X_val_imp[cols])
    ridge_models[name] = model
    ridge_results.append({
        "Feature_Set": name,
        "N_Features": len(cols),
        **metrics(y_val, pred_val)
    })

ridge_comparison = pd.DataFrame(ridge_results).sort_values("RMSE").reset_index(drop=True)
display(ridge_comparison)

best_feature_set = ridge_comparison.loc[0, "Feature_Set"]
fixed_features = feature_sets[best_feature_set]
print("Selected feature set for Model Comparison:", best_feature_set)
print("Fixed feature count:", len(fixed_features))

## 7. Experiment 2 — Compare five models on the same fixed feature set

为了公平比较，五个模型使用完全相同的 `fixed_features`。每个模型只尝试少量代表性参数组合；根据 Validation RMSE 选择模型和参数。所有模型均为 CPU 模型，不需要 GPU。

In [ ]:
Xtr = X_train_imp[fixed_features]
Xva = X_val_imp[fixed_features]
Xte = X_test_imp[fixed_features]

model_configs = {
    "Ridge": (
        lambda p: make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            Ridge(alpha=p["alpha"])
        ),
        [{"alpha": 1.0}, {"alpha": 10.0}, {"alpha": 50.0}]
    ),
    "RandomForest": (
        lambda p: RandomForestRegressor(
            n_estimators=180, max_features=p["max_features"],
            min_samples_leaf=p["min_samples_leaf"],
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        [
            {"max_features": 0.7, "min_samples_leaf": 3},
            {"max_features": 1.0, "min_samples_leaf": 5}
        ]
    ),
    "ExtraTrees": (
        lambda p: ExtraTreesRegressor(
            n_estimators=220, max_features=p["max_features"],
            min_samples_leaf=p["min_samples_leaf"],
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        [
            {"max_features": 0.7, "min_samples_leaf": 2},
            {"max_features": 1.0, "min_samples_leaf": 5}
        ]
    ),
    "HistGradientBoosting": (
        lambda p: HistGradientBoostingRegressor(
            max_iter=p["max_iter"], learning_rate=p["learning_rate"],
            max_leaf_nodes=p["max_leaf_nodes"],
            l2_regularization=1.0, random_state=RANDOM_STATE
        ),
        [
            {"max_iter": 150, "learning_rate": 0.05, "max_leaf_nodes": 15},
            {"max_iter": 250, "learning_rate": 0.04, "max_leaf_nodes": 25}
        ]
    ),
    "GradientBoosting": (
        lambda p: GradientBoostingRegressor(
            n_estimators=p["n_estimators"], learning_rate=p["learning_rate"],
            max_depth=p["max_depth"], loss="huber", random_state=RANDOM_STATE
        ),
        [
            {"n_estimators": 120, "learning_rate": 0.05, "max_depth": 2},
            {"n_estimators": 180, "learning_rate": 0.04, "max_depth": 2}
        ]
    )
}

tuning_rows = []
best_estimators = {}

for model_name, (build_model, param_list) in model_configs.items():
    best_rmse = np.inf
    best_model, best_params = None, None

    for params in param_list:
        model = build_model(params)
        model.fit(Xtr, y_train)
        pred = model.predict(Xva)
        rmse = mean_squared_error(y_val, pred) ** 0.5

        if rmse < best_rmse:
            best_rmse = rmse
            best_model = model
            best_params = params

    pred_val = best_model.predict(Xva)
    tuning_rows.append({
        "Model": model_name,
        "N_Features": len(fixed_features),
        **metrics(y_val, pred_val),
        "Best_Params": str(best_params)
    })
    best_estimators[model_name] = best_model

model_comparison = pd.DataFrame(tuning_rows).sort_values("RMSE").reset_index(drop=True)
display(model_comparison)

## 8. Final Test evaluation — evaluate the validation winner once

Test 不参与因子筛选、因子集选择、参数选择或模型选择。这里仅对 Validation 最优模型做一次最终测试。

In [ ]:
best_model_name = model_comparison.loc[0, "Model"]
# Refit selected configuration on Train + Validation, then evaluate Test once.
from sklearn.base import clone
final_model = clone(best_estimators[best_model_name])
X_train_val = pd.concat([Xtr, Xva])
y_train_val = pd.concat([y_train, y_val])
final_model.fit(X_train_val, y_train_val)
test_prediction = final_model.predict(Xte)

final_test_metrics = pd.DataFrame(
    [metrics(y_test, test_prediction)],
    index=[best_model_name]
)
display(final_test_metrics)

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(y_test.index, y_test, label="Actual Thermal", lw=0.7)
ax.plot(y_test.index, test_prediction, label="Predicted Thermal", lw=0.7)
ax.set_title(f"Thermal Generation — Test: {best_model_name}")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

residual = y_test - test_prediction
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(residual, bins=50, kde=True, ax=axes[0])
axes[0].set_title("Test residual distribution")
axes[1].scatter(test_prediction, residual, s=5, alpha=0.25)
axes[1].axhline(0, ls="--", color="black")
axes[1].set_title("Residual vs predicted")
axes[1].set_xlabel("Predicted Thermal (MW)")
axes[1].set_ylabel("Residual (MW)")
plt.tight_layout()
plt.show()

## 9. Interpretation and reporting

报告时建议按以下顺序讨论：

1. **Feature screening：** 候选特征数量、Train 缺失率筛选后数量、去相关后保留数量；说明 Pearson 相关只用于快速筛选。
2. **Feature-set experiment：** 在相同 Ridge 下比较 Small / Medium / Large 的 Validation RMSE、MAE、R²，判断增加特征是否带来增量。
3. **Model experiment：** 固定同一因子集，比较五种模型及各自验证集最优参数，避免因子集不同导致不公平比较。
4. **Test：** 只报告最终模型在 Test 上的结果；不要根据 Test 反复调整特征或参数。
5. **Economic interpretation：** Wind/Solar 是同期已知输入；模型捕捉的是条件预测关系，不应将相关性或 SHAP 解释为因果效应。

**重要限制：** 当前任务是同期 nowcast（已知时点 t 的 Wind / Solar，预测 Thermal_t），不是提前预测未来小时。若改成 t+1 或 t+24 预测，必须重新设计输入时点和所有 shift 规则。

## 10. Sanity checks

In [ ]:
# No target-derived feature names and no target column in X.
assert "Thermal" not in X_raw.columns
assert all("Thermal" not in c for c in X_raw.columns)
assert X_train_imp.index.max() < X_val_imp.index.min()
assert X_val_imp.index.max() < X_test_imp.index.min()
assert set(fixed_features).issubset(X_train_imp.columns)

print("Candidate features:", X_raw.shape[1])
print("De-correlated features:", len(selected))
print("Feature sets:", {k: len(v) for k, v in feature_sets.items()})
print("Fixed model-comparison feature set:", best_feature_set, len(fixed_features))
print("Best validation model:", best_model_name)
print("Final test metrics:")
display(final_test_metrics)